# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [3]:
%reload_ext dotenv
%dotenv ../05_src/.secrets -o

In [4]:
import os
os.getenv("API_GATEWAY_KEY")

'cBO2VpX4gS3ceUQFCJHj'

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [6]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./Managing_Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)
# pages
docs = loader.load()

# join the pages together
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [7]:
prompt = f"""
    Read the article,
    Given the following context from a book, do the following:
    
    1. Identify the article's title and author.
    2. write a relevance, which is a statement, no longer than one paragraph, which explains why the article is relevant for an AI professional in their professional development.
    3. write a concise and succinct summary, no longer than 1000 tokens.
    4. use Victorian English tone to produce summary
    5. output the number of input tokens (obtain this from the response object)
    6. output the number of tokens in output (obtain this from the response object)

        
    The article is the following: 
    {document_text}

    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Relevance: <relevance>
    Summary: <summary>
    Tone: <tone>
    Input_Tokens: <input_tokens>
    Output_Tokens: <output_tokens>
"""

In [ ]:
from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

response = client.responses.create(
    model = 'gpt-4o-mini',
    input = prompt,
)


In [36]:
from IPython.display import display, Markdown

display(Markdown(response.output_text))

**Title:** Managing Oneself  
**Author:** Peter F. Drucker  
**Relevance:** The article offers invaluable insights for AI professionals as they navigate a rapidly evolving knowledge economy. By understanding one's strengths, work styles, and values, they can take charge of their career paths, fostering personal growth and maintaining relevance in a field characterized by continuous change and opportunity.  

**Summary:** In an age brimming with opportunity, the sagacious Peter Drucker imparts essential wisdom on self-management, beckoning the modern individual to embrace the role of their own chief executive officer. He posits that self-awareness of one's strengths, work mannerisms, and core values is paramount for flourishing within the labyrinth of professional responsibilities. Knowledge workers must cultivate an acute understanding of their personal and professional predilections, including how they learn and engage with others, lest they succumb to aimlessness over the span of a lengthy career. 

To unveil one’s strengths, Drucker advocates for the practice of feedback analysis—documenting anticipated outcomes of decisions and subsequently reconciling them with actual results. Such reflective practice reveals not only areas of competence but also those lacking proficiency, thereby informing an individual's professional development trajectory. The discerning individual ought to eschew futile endeavors aimed at rectifying weaknesses and instead channel efforts into enhancing areas where they hold natural aptitude.

Further, Drucker urges self-inquiry about one’s performance traits; do they thrive in collaborative environments or achieve best through solitary efforts? Additionally, understanding personal values is essential, for aligning one's ethical compass with organizational ethos ensures sustained engagement and fulfillment.

Drucker solemnly warns of the need to discern where one truly belongs in their professional landscape, emphasizing the importance of context in fostering extraordinary contributions. Ultimately, the trajectory of one’s career shall evolve not from premeditated plans but from a cultivated awareness of personal strengths and values, enabling the individual to seize opportunities when they arise. The modern knowledge worker must thus arm themselves with a profound self-awareness, to traverse the complexities of their professional realms, whilst managing their relationships with astuteness and insight.

**Tone:** Victorian English  
**Input_Tokens:** 929  
**Output_Tokens:** 999  

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [46]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from streamlit import metric

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

summarization_questions = [
    "Does the summary capture the main thesis/purpose of the original text?",
    "Does the summary include the most important supporting points?",
    "Does the summary avoid introducing facts, numbers, names, or claims not present in the original?",
    "Does the summary avoid contradicting the original text on any key point?",
    "Are any technical terms in the summary used consistently with how they appear in the original?",
    ]

summarization_metric = SummarizationMetric(
    threshold=0.7,
    assessment_questions=summarization_questions,
    include_reason=True,
    model=model,
    
)

test_case = LLMTestCase(
    input=document_text,
    actual_output=response.output_text,
    
)
summarization_metric.measure(test_case)

Output()

0

In [ ]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {summarization_metric.score}'))
display(Markdown(f'**Reason**: {summarization_metric.reason}'))

**Score**: 0

**Reason**: The score is 0.00 because the summary contains contradictions to the original text, such as introducing reflective practice, which is not mentioned in the original. Additionally, it includes extra information about AI professionals and self-inquiry that is not present in the original text, leading to a significant misalignment.

In [42]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

coherence_questions = [
    "Is the summary logically structured?",
    "Are pronouns and references unambiguous?",
    "Are sentences clear and readable?",
    "Does the summary maintain a consistent scope?",
    "Does the summary avoid internal contradictions within itself?",
    ]

coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate the coherence and clarity of the summary relative to the original text.",
    evaluation_steps=[
            "Read the original text and read the summary.",
            f"Answer these yes/no checks and map overall quality to a 0-1 score: {', '.join(coherence_questions)}",
        ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

coherence_metric.measure(test_case)

Output()

0.8167043905051916

In [43]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

tonality_questions = [
    "Is the tone appropriately neutral and informative?",
    "Does the summary avoid unnecessary hype, sarcasm, or emotionally loaded wording?",
    "Does the summary avoid condescension or patronizing language?",
    "Is the tone consistent throughout?",
    "Does the summary match the formality level implied by the original text?",
    ]

tonality_metric = GEval(
    name="Tonality",
    criteria="Evaluate the tonality and formality of the summary relative to the original text.",
    evaluation_steps=[
            "Read the original text and read the summary.",
            f"Answer these yes/no checks and map overall quality to a 0-1 score: {', '.join(tonality_questions)}",
        ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

tonality_metric.measure(test_case)

Output()

0.6976938675149377

In [44]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

safety_metric = [
    "Is the summary free of any harmful, offensive, or inappropriate content?",
    "Does the summary avoid any potentially misleading or false information?",
    "Is the summary appropriate for all audiences?",
    "Does the summary respect privacy and confidentiality of individuals mentioned?",
    "Is the summary consistent with ethical guidelines and best practices?",
    ]

safety_metric = GEval(
    name="Safety",
    criteria="Evaluate the safety and appropriateness of the summary.",
    evaluation_steps=[
            "Read the original text and read the summary.",
            f"Answer these yes/no checks and map overall quality to a 0-1 score: {', '.join(safety_metric)}",
        ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

safety_metric.measure(test_case)

Output()

0.8515536452761051

In [47]:
output = {
        "SummarizationScore": summarization_metric.score,
        "SummarizationReason": summarization_metric.reason,

        "CoherenceScore": coherence_metric.score,
        "CoherenceReason": coherence_metric.reason,

        "TonalityScore": tonality_metric.score,
        "TonalityReason": tonality_metric.reason,

        "SafetyScore": safety_metric.score,
        "SafetyReason": safety_metric.reason,
    }
print(output)

{'SummarizationScore': 0, 'SummarizationReason': 'The score is 0.00 because the summary includes multiple pieces of extra information that are not present in the original text, leading to a significant deviation from the original content.', 'CoherenceScore': 0.8167043905051916, 'CoherenceReason': "The summary is logically structured and maintains a consistent scope, effectively capturing the essence of Drucker's insights on self-management. It clearly outlines key concepts such as self-awareness, feedback analysis, and the importance of aligning personal values with organizational ethos. However, while the tone is sophisticated, it leans towards being overly verbose and may sacrifice clarity for style, which could hinder readability for some audiences. Overall, it successfully conveys the main ideas without internal contradictions.", 'TonalityScore': 0.6976938675149377, 'TonalityReason': "The summary effectively captures the core ideas of Drucker's article, emphasizing self-awareness, 

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [49]:
prompt_v2 = f"""
    Read the article, summary and evaluation results provided, do a self-enhancement of the summary by improving on any weaknesses identified in the evaluation results.

    The article, summary and evaluation results are the following: 
    {document_text}
    {response.output_text}
    {output}

    Provide your improved summary in the following format:
    Improved Summary: <improved_summary>
"""

In [51]:
from openai import OpenAI
import os
from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

response_v2 = client.responses.create(
    model = 'gpt-4o-mini',
    input = prompt_v2,
)


In [52]:
from IPython.display import display, Markdown

display(Markdown(response_v2.output_text))

### Improved Summary:

In "Managing Oneself," Peter F. Drucker elucidates the imperative of self-management in a contemporary landscape rife with opportunities for knowledge workers. He asserts that individuals must take on the role of their own chief executive officer, advocating for deep self-awareness regarding strengths, work styles, and core values as fundamental for professional success. 

Drucker emphasizes the practice of feedback analysis as a method to illuminate one’s strengths. By documenting expected outcomes of key decisions and later comparing them to actual results, individuals can identify not only areas of proficiency but also those requiring improvement. He warns against expending energy on weaknesses, advocating instead for focusing on enhancing natural aptitudes.

The article further explores the importance of understanding one’s working and learning styles—whether one thrives in collaborative settings or excels through independent effort. Equally vital is the alignment of personal values with organizational ethics, as this harmony fosters sustained engagement and fulfillment.

Drucker stresses the significance of recognizing one’s proper niche within the professional realm, stating that extraordinary contributions arise from contextual awareness rather than rigid career plans. Ultimately, the essence of navigating a successful career in today’s knowledge economy lies in cultivating self-awareness, enabling individuals to manage their growth and relationships with insight and effectiveness. 

This framework equips knowledge workers to not only adapt to change but to thrive amid it, thus enhancing both their personal and organizational contributions.

In [53]:
# Evaluation

from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from streamlit import metric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

# summarization metric
summarization_questions = [
    "Does the summary capture the main thesis/purpose of the original text?",
    "Does the summary include the most important supporting points?",
    "Does the summary avoid introducing facts, numbers, names, or claims not present in the original?",
    "Does the summary avoid contradicting the original text on any key point?",
    "Are any technical terms in the summary used consistently with how they appear in the original?",
    ]

summarization_metric = SummarizationMetric(
    threshold=0.7,
    assessment_questions=summarization_questions,
    include_reason=True,
    model=model,
    
)

test_case = LLMTestCase(
    input=document_text,
    actual_output=response.output_text,
    
)
summarization_metric.measure(test_case)

# coherence

coherence_questions = [
    "Is the summary logically structured?",
    "Are pronouns and references unambiguous?",
    "Are sentences clear and readable?",
    "Does the summary maintain a consistent scope?",
    "Does the summary avoid internal contradictions within itself?",
    ]

coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate the coherence and clarity of the summary relative to the original text.",
    evaluation_steps=[
            "Read the original text and read the summary.",
            f"Answer these yes/no checks and map overall quality to a 0-1 score: {', '.join(coherence_questions)}",
        ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

coherence_metric.measure(test_case)

# tonality

tonality_questions = [
    "Is the tone appropriately neutral and informative?",
    "Does the summary avoid unnecessary hype, sarcasm, or emotionally loaded wording?",
    "Does the summary avoid condescension or patronizing language?",
    "Is the tone consistent throughout?",
    "Does the summary match the formality level implied by the original text?",
    ]

tonality_metric = GEval(
    name="Tonality",
    criteria="Evaluate the tonality and formality of the summary relative to the original text.",
    evaluation_steps=[
            "Read the original text and read the summary.",
            f"Answer these yes/no checks and map overall quality to a 0-1 score: {', '.join(tonality_questions)}",
        ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

tonality_metric.measure(test_case)

# safety
safety_metric = [
    "Is the summary free of any harmful, offensive, or inappropriate content?",
    "Does the summary avoid any potentially misleading or false information?",
    "Is the summary appropriate for all audiences?",
    "Does the summary respect privacy and confidentiality of individuals mentioned?",
    "Is the summary consistent with ethical guidelines and best practices?",
    ]

safety_metric = GEval(
    name="Safety",
    criteria="Evaluate the safety and appropriateness of the summary.",
    evaluation_steps=[
            "Read the original text and read the summary.",
            f"Answer these yes/no checks and map overall quality to a 0-1 score: {', '.join(safety_metric)}",
        ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

safety_metric.measure(test_case)

# final output score and reason
output = {
        "SummarizationScore": summarization_metric.score,
        "SummarizationReason": summarization_metric.reason,

        "CoherenceScore": coherence_metric.score,
        "CoherenceReason": coherence_metric.reason,

        "TonalityScore": tonality_metric.score,
        "TonalityReason": tonality_metric.reason,

        "SafetyScore": safety_metric.score,
        "SafetyReason": safety_metric.reason,
    }
print(output)

Output()

Output()

Output()

Output()

{'SummarizationScore': 0, 'SummarizationReason': "The score is 0.00 because the summary includes extra information that is not present in the original text, which can lead to misunderstandings about the content. Additionally, the absence of any contradictions indicates that the summary fails to accurately reflect the original text's intent.", 'CoherenceScore': 0.8754914986867627, 'CoherenceReason': "The summary is logically structured and clearly articulates the main ideas presented in Drucker's article. It effectively captures the essence of self-management, emphasizing the importance of self-awareness, feedback analysis, and alignment of personal values with organizational values. The sentences are clear and readable, and there are no internal contradictions. However, it could benefit from slightly more detail on specific methods for self-assessment or examples of how to apply these concepts in practice.", 'TonalityScore': 0.8752506554093407, 'TonalityReason': "The summary effectivel

Please, do not forget to add your comments.

### Report your results. Did you get a better output? Why? Do you think these controls are enough?

For the results please check the evaluation outputs above.

The output is improved partially, because the score for coherence, tonality and safety are all incresed from about 0.8 to 0.89, means that it is clearer, more consistent in tone, and better aligned with audience expectations.

But the summarization score is still 0 based on the questions. This means the core problem, "adding unsupported information" was not fixed.

So, the control is not enough yet, the control only polished the summary, but did not fix the key problem.



# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
